In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# ---------- Paths ----------
def find_base_dir():
    env = os.environ.get("PROJECT_DIR")
    if env and (Path(env) / "data" / "raw").exists():
        return Path(env)

    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "data" / "raw").exists():
            return p

    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        chosen = filedialog.askdirectory(title="Select the project folder (the one containing data/raw)")
        root.destroy()
    except Exception:
        chosen = input("Paste the project folder path (the one containing data/raw): ").strip().strip('"')

    if chosen and (Path(chosen) / "data" / "raw").exists():
        return Path(chosen)
    raise FileNotFoundError("data/raw not found in the selected folder.")

BASE_DIR = find_base_dir()
RAW_DIR = BASE_DIR / "data" / "raw"
OUT_DIR = BASE_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("BASE_DIR:", BASE_DIR)

def safe_save(save_fn, filename):
    """Try OUT_DIR first; if the OS rejects the write, fall back to the home folder."""
    try:
        save_fn(OUT_DIR / filename)
        print("Saved:", OUT_DIR / filename)
    except OSError as e:
        fallback = Path.home() / "part1c_outputs"
        fallback.mkdir(exist_ok=True)
        save_fn(fallback / filename)
        print("Could not write to", OUT_DIR, "-", e)
        print("Saved to", fallback / filename, "instead.")

# ---------- Data ----------
lt = pd.read_csv(RAW_DIR / "life_table_inputs.csv").copy()

# 1. Central death rate mx and death probability qx
lt["mx"] = lt["deaths"] / lt["person_years"]
lt["nx"] = lt["interval_years"]
lt["qx"] = np.where(
    lt["nx"].isna(),
    1.0,
    (lt["nx"] * lt["mx"]) / (1.0 + (lt["nx"] - lt["ax"]) * lt["mx"])
)

# 2. Radix = 100,000; recursive lx, dx, Lx
radix = 100000.0
lx_list, dx_list, Lx_list = [], [], []
curr_l = radix

for _, row in lt.iterrows():
    lx_list.append(curr_l)
    dx = curr_l * row["qx"]
    dx_list.append(dx)
    if pd.isna(row["nx"]):
        Lx = curr_l / row["mx"]
        next_l = 0.0
    else:
        next_l = curr_l - dx
        Lx = row["nx"] * next_l + row["ax"] * dx
    Lx_list.append(Lx)
    curr_l = next_l

lt["lx"] = lx_list
lt["dx"] = dx_list
lt["Lx"] = Lx_list
lt["Tx"] = lt["Lx"].iloc[::-1].cumsum().iloc[::-1]
lt["ex"] = lt["Tx"] / lt["lx"]

# 3. Numerical checks
assert np.isclose(lt["dx"].sum(), radix), "Sum of dx must equal radix"
assert np.isclose(lt["Lx"].sum(), lt.loc[0, "Tx"]), "Sum of Lx must equal T0"
L0_check = 5.0 * lt.loc[1, "lx"] + 1.2 * lt.loc[0, "dx"]
assert np.isclose(L0_check, lt.loc[0, "Lx"]), "Row 0 L0 hand check failed"

# Show the table first, then save
display(lt[["age_start", "nx", "mx", "ax", "qx", "lx", "dx", "Lx", "Tx", "ex"]])
safe_save(lambda p: lt.to_csv(p, index=False), "part1c_abridged_life_table.csv")

print(f"\nNumerical check passed: sum(Lx) = T0 = {lt.loc[0, 'Tx']:,.2f}; L0 = {L0_check:,.2f}")
print(f"e0 = {lt.loc[0, 'ex']:.4f} years | e65 = {lt.loc[lt['age_start'] == 65, 'ex'].values[0]:.4f} years")

BASE_DIR: C:\Users\user\Documents\GitHub\Group-4-Assignment-2-Two-


,age_start,nx,mx,ax,qx,lx,dx,Lx,Tx,ex
0,0,5.0,0.008000,1.2,0.038820,100000.000000,3881.987578,485248.447205,7.310913e+06,73.109127
1,5,5.0,0.000900,2.5,0.004490,96118.012422,431.560046,479511.161997,6.825664e+06,71.013373
2,10,5.0,0.000600,2.5,0.002996,95686.452377,286.629413,477715.688350,6.346153e+06,66.322379
3,15,5.0,0.000800,2.5,0.003992,95399.822964,380.837617,476047.020776,5.868437e+06,61.514133
4,20,5.0,0.001105,2.5,0.005509,95018.985347,523.421137,473786.373892,5.392390e+06,56.750663
5,25,5.0,0.001500,2.5,0.007472,94495.564210,706.068973,470712.648618,4.918604e+06,52.051163
6,30,5.0,0.002000,2.5,0.009950,93789.495237,933.228808,466614.404165,4.447891e+06,47.424196
7,35,5.0,0.002700,2.5,0.013409,92856.266429,1245.154802,461168.445139,3.981277e+06,42.875695
8,40,5.0,0.003506,2.5,0.017377,91611.111627,1591.936047,454075.718018,3.520109e+06,38.424471
9,45,5.0,0.004800,2.5,0.023715,90019.175580,2134.842109,444758.772630,3.066033e+06,34.059775


Saved: C:\Users\user\Documents\GitHub\Group-4-Assignment-2-Two-\outputs\part1c_abridged_life_table.csv

Numerical check passed: sum(Lx) = T0 = 7,310,912.73; L0 = 485,248.45
e0 = 73.1091 years | e65 = 18.0962 years
